# Practice: How Much Does Preprocessing Change Your Result?

**Computational Text Analysis Pack · Module 03 — Preparing Text**

*Technique:* One finding, five preprocessing pipelines, and the spread between them

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/01-preparing-text/03c_practice_preprocessing.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **If five researchers computed the same keyword list from the same corpus,
> making ordinary and defensible preprocessing choices, how much would their
> answers differ?**

Preprocessing is usually a sentence in a methods section. This notebook treats
it as the variable it is.

### How to use this notebook

This is a worked case, not a tutorial. Run it once to see the shape of the
argument, then change the question at the top to one of your own and run it
again. The cells are written so that swapping the question is the only edit you
need.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
import re
import numpy as np

ENGLISH_STOP = {"the","a","an","and","or","but","of","to","in","on","for","with",
                "is","was","are","were","be","been","it","its","this","that",
                "as","at","by","from","has","have","had","he","she","they","we"}

PIPELINES = {
    "unicode-aware, keep all":  lambda t: re.findall(r"[\w\u0300-\u036f]+", t.lower()),
    "ascii-only [a-z]+":        lambda t: re.findall(r"[a-z]+", t.lower()),
    "drop English stopwords":   lambda t: [w for w in re.findall(r"[\w\u0300-\u036f]+", t.lower())
                                           if w not in ENGLISH_STOP],
    "min length 4":             lambda t: [w for w in re.findall(r"[\w\u0300-\u036f]+", t.lower())
                                           if len(w) >= 4],
    "keep case":                lambda t: re.findall(r"[\w\u0300-\u036f]+", t),
}
target = df[df["category"] == "health"]["text"].head(300).tolist()
reference = df[df["category"] != "health"]["text"].head(300).tolist()
print(f"target: {len(target)} health documents; reference: {len(reference)} others")

In [ ]:
def log_likelihood(a, b, na, nb):
    if a == 0 and b == 0: return 0.0
    ea, eb = na*(a+b)/(na+nb), nb*(a+b)/(na+nb)
    ll = 0.0
    if a: ll += a*np.log(a/ea)
    if b: ll += b*np.log(b/eb)
    return 2*ll

results = {}
for name, tok in PIPELINES.items():
    ca, cb = Counter(), Counter()
    for t in target: ca.update(tok(str(t)))
    for t in reference: cb.update(tok(str(t)))
    na, nb = sum(ca.values()), sum(cb.values())
    scored = [(w, log_likelihood(ca[w], cb[w], na, nb))
              for w in ca if ca[w] + cb[w] >= 15]
    scored.sort(key=lambda kv: -kv[1])
    results[name] = [w for w, _ in scored[:20]]
    print(f"{name:<26}{', '.join(results[name][:8])}")

In [ ]:
# How much do the five keyword lists actually agree?
names = list(results)
print(f"{'':<26}" + "".join(f"{n[:11]:>13}" for n in names))
print("-" * (26 + 13*len(names)))
for a in names:
    row = "".join(f"{len(set(results[a]) & set(results[b])):>13}" for b in names)
    print(f"{a:<26}{row}")
print("\n(cells = shared words in the top 20)")

common = set.intersection(*[set(v) for v in results.values()])
union = set.union(*[set(v) for v in results.values()])
print(f"\nwords ALL five pipelines agree on : {len(common)} of 20")
print(f"words appearing in at least one   : {len(union)}")
print(f"agreement rate                    : {len(common)/len(union):.1%}")

plt.figure(figsize=(7, 3.6))
counts = Counter(w for v in results.values() for w in v)
byn = Counter(counts.values())
plt.bar([str(k) for k in sorted(byn)], [byn[k] for k in sorted(byn)], color="steelblue")
plt.xlabel("number of pipelines a word appears in"); plt.ylabel("words")
plt.title("Most keywords are pipeline-dependent")
plt.tight_layout()

## What this licenses

**Defensible:** "The keyword list for health coverage is sensitive to
preprocessing; only the words appearing under all five pipelines should be
treated as robust."

**Not defensible:** any claim resting on a word that appeared under one
tokenisation. And that is most of them.

The practical consequence is a reporting rule: **state the tokeniser, the
stopword list, the minimum frequency and the case handling**, or your keyword
list cannot be reproduced. The `ascii-only` row is the one to look at hardest —
it is the most common default in tutorials, and on this corpus it silently
discards the languages the course exists to study.

### 🔧 Your turn 1

Add a sixth pipeline that stems the tokens (module 14a) and see whether it agrees with any of the others.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Stemming usually produces the most divergent list of all, because it merges
words the others keep separate and its output often is not a word — so the
"keywords" become stems a reader cannot look up. That is Schofield and Mimno's
warning arriving from the keyness direction rather than the topic-model one, and
it is why the robust-intersection approach above is worth more than any single
pipeline's ranking.

</details>